In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:

df=pd.read_csv(r"C:\Users\EVO TECH\Downloads\New folder\Inventory Management E-Grocery - InventoryData.csv")
print(df.head())

In [ ]:
print(list(df.columns))

In [ ]:
selected_columns = [
    'SKU_ID', 'SKU_Name', 'Category', 'ABC_Class', 
    'Received_Date', 'Expiry_Date', 
    'Quantity_On_Hand', 'Damaged_Qty', 
    'Unit_Cost_USD', 'Total_Inventory_Value_USD', 'FIFO_FEFO'
]

file_path = r"C:\Users\EVO TECH\Downloads\New folder\Inventory Management E-Grocery - InventoryData.csv"
df = pd.read_csv(file_path, usecols=selected_columns)
df['Received_Date'] = pd.to_datetime(df['Received_Date'], errors='coerce')
df['Expiry_Date'] = pd.to_datetime(df['Expiry_Date'], errors='coerce')
print(df.head())

In [ ]:
df.info()
print(df.isnull().sum())

In [ ]:

df['Unit_Cost_USD'] = df['Unit_Cost_USD'].str.replace('$', '').str.replace(',', '').astype(float)
df['Total_Inventory_Value_USD'] = df['Total_Inventory_Value_USD'].str.replace('$', '').str.replace(',', '').astype(float)
df['Shelf_Life_Days'] = (df['Expiry_Date'] - df['Received_Date']).dt.days
print(df[['Unit_Cost_USD', 'Total_Inventory_Value_USD', 'Shelf_Life_Days']].dtypes)

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=df, 
    x='Received_Date', 
    y='Expiry_Date', 
    hue='Category', 
    palette='Set2', 
    alpha=0.8
)

plt.title('ﺎﻬﺘﻴﺣﻼﺻ ءﺎﻬﺘﻧﺍ ﺦﻳﺭﺎﺗﻭ تﺎﺠﺘﻨﻤﻟﺍ مﻼﺘﺳﺍ ﺦﻳﺭﺎﺗ ﻦﻴﺑ ﺔﻗﻼﻌﻟﺍ', fontsize=14, fontweight='bold')
plt.xlabel('مﻼﺘﺳﻻﺍ ﺦﻳﺭﺎﺗ (Received Date)', fontsize=12)
plt.ylabel('ﺔﻴﺣﻼﺼﻟﺍ ءﺎﻬﺘﻧﺍ ﺦﻳﺭﺎﺗ (Expiry Date)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
print(df.groupby('Category')['FIFO_FEFO'].value_counts())

In [ ]:
fefo_products=df[df['FIFO_FEFO']=='FEFO']
print(fefo_products.head())

In [ ]:

df['Shelf_Life_Days'] = (df['Expiry_Date'] - df['Received_Date']).dt.days
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(
    ax=axes[0],
    data=df, 
    x='FIFO_FEFO', 
    y='Shelf_Life_Days', 
    palette='Set2'
)
axes[0].set_title('فﺮﻟﺍ ﻰﻠﻋ ةﺪﻤﻟﺍﻭ (FIFO /FEFO) فﺮﺼﻟﺍ مﺎﻈﻧ ﻦﻴﺑ ﺔﻗﻼﻌﻟﺍ ', fontsize=14, fontweight='bold')
axes[0].set_xlabel('ﺞﺘﻨﻤﻟﺍ جﻭﺮﺧ مﺎﻈﻧ', fontsize=12)
axes[0].set_ylabel('مﺎﻳﻻﺎﺑ فﺮﻟﺍ ﻰﻠﻋ ةﺪﻤﻟﺍ ', fontsize=12)
axes[0].grid(axis='y', linestyle='--', alpha=0.6)

damaged_summary = df.groupby('FIFO_FEFO')['Damaged_Qty'].sum().reset_index()

sns.barplot(
    ax=axes[1],
    data=damaged_summary, 
    x='FIFO_FEFO', 
    y='Damaged_Qty', 
    palette='Reds'
)
axes[1].set_title('فﺮﺼﻟﺍ مﺎﻈﻧ ﻰﻠﻋ ًءﺎﻨﺑ ﺔﻔﻟﺎﺘﻟﺍ تﺎﻴﻤﻜﻟﺍ ﻲﻟﺎﻤﺟﺇ', fontsize=14, fontweight='bold')
axes[1].set_xlabel('ﺞﺘﻨﻤﻟﺍ جﻭﺮﺧ مﺎﻈﻧ', fontsize=12)
axes[1].set_ylabel('ﺔﻔﻟﺎﺘﻟﺍ تﺎﻴﻤﻜﻟﺍ ﻲﻟﺎﻤﺟﺇ', fontsize=12)

for index, row in damaged_summary.iterrows():
    axes[1].text(index, row.Damaged_Qty, f'{int(row.Damaged_Qty)}', color='black', ha="center", va="bottom", fontweight='bold')
    
plt.tight_layout()
plt.show()

In [ ]:

recent_date = df['Received_Date'].max() - pd.Timedelta(days=60)
fast_moving_range = df[(df['Shelf_Life_Days'] <= 30) & (df['Received_Date'] >= recent_date)]

fast_moving_range['Damage_Ratio'] = (fast_moving_range['Damaged_Qty'] / fast_moving_range['Quantity_On_Hand']) * 100

report = fast_moving_range.sort_values(by='Damage_Ratio', ascending=False)

plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=report,
    x='Shelf_Life_Days',
    y='Damage_Ratio',
    hue='Category',
    size='Damaged_Qty', 
    sizes=(50, 400),
    alpha=0.7
)

plt.title('فﺮﻟﺍ ﻰﻠﻋ ةﺪﻤﻟﺍ ﻞﺑﺎﻘﻣ ﻒﻠﺘﻟﺍ ﺔﺒﺴﻧ( ًﺎﺒﻳﺮﻗ ﺔﻤﻠﺘﺴﻤﻟﺍﻭ ﻒﻠﺘﻟﺍ ﺔﻌﻳﺮﺳ تﺎﺠﺘﻨﻤﻟﺍ ﻞﻴﻠﺤﺗ)', fontsize=14)
plt.xlabel('مﺎﻳﺃ( فﺮﻟﺍ ﻰﻠﻋ ةﺪﻤﻟﺍ)', fontsize=12)
plt.ylabel('ﻒﻠﺘﻟﺍ ﺔﺒﺴﻧ (%)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


print("🚨 المنتجات في الرينج ده اللي فيها أعلى نسبة تلف:")
print(report[['SKU_Name', 'Category', 'Shelf_Life_Days', 'Damage_Ratio']].head(10))

In [ ]:

plt.figure(figsize=(14, 7))

sns.scatterplot(
    data=df, 
    x='Received_Date', 
    y='Expiry_Date', 
    hue='Category', 
    palette='viridis', 
    alpha=0.6, 
    s=70
)

plt.plot([df['Received_Date'].min(), df['Received_Date'].max()], 
         [df['Received_Date'].min(), df['Received_Date'].max()], 
         color='red', linestyle='--', label='ءﺎﻬﺘﻧﻻﺍ = مﻼﺘﺳﻻﺍ( يﻭﺎﺴﺘﻟﺍ ﻂﺧ)')


plt.title('ءﺎﻬﺘﻧﻻﺍ ﺦﻳﺭﺎﺗﻭ مﻼﺘﺳﻻﺍ ﺦﻳﺭﺎﺗ ﻦﻴﺑ ﺔﻴﻨﻣﺰﻟﺍ ﺔﻗﻼﻌﻟﺍ', fontsize=16, fontweight='bold')
plt.xlabel('ﺞﺘﻨﻤﻟﺍ مﻼﺘﺳﺍ ﺦﻳﺭﺎﺗ', fontsize=12)
plt.ylabel('ﺔﻴﺣﻼﺼﻟﺍ ءﺎﻬﺘﻧﺍ ﺦﻳﺭﺎﺗ', fontsize=12)
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()